In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Any, Dict, Tuple, Optional

import sys
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils.json_utils import load_jsonl
from src.llm.summary_schema import parse_structured_summary

from src.eval.embedding_backend import SentenceTransformersEmbedder
from src.eval.summary_metrics import (
    prf1_set,
    context_similarity,
    match_actions_by_description,
    micro_prf,
    overall_auto_score,
)

GOLD_PATH = Path("../data/processed/gold_summaries_final.json")

PRED_PATHS = {
    "mistral_large": Path("../data/predictions/mistral_large.jsonl"),
    "ollama_llama3.1_8b": Path("../data/predictions/ollama_llama3.1_8b.jsonl"),
    "ollama_mistral_7b": Path("../data/predictions/ollama_mistral_7b.jsonl"),
    "ollama_qwen2.5_3b": Path("../data/predictions/ollama_qwen2.5_3b.jsonl"),
    "openai_gpt4o": Path("../data/predictions/openai_gpt4o.jsonl"),
}

ACTION_THRESHOLDS = [0.75, 0.80]

# Embedding backend (по умолчанию HF на MPS/CPU)
# device: "mps" (mac), "cpu", "cuda"
EMBEDDER = SentenceTransformersEmbedder(model_name="intfloat/multilingual-e5-small", device="mps")


In [16]:
def extract_dialogue_id(obj: Dict[str, Any]) -> str:
    did = obj.get("dialogue_id")
    if did is None:
        raise ValueError("No dialogue_id field in object")
    return str(did)


def extract_dialogue_text(obj: Dict[str, Any]) -> str:
    d = obj.get("dialogue")
    if isinstance(d, str):
        return d
    return ""


EMPTY_SUMMARY = {"Context": "", "Decisions": [], "Actions": [], "Questions": []}

def _is_effectively_empty_summary(s: Dict[str, Any]) -> bool:
    """Heuristic: treat as empty if all fields are empty / blank."""
    if not isinstance(s, dict):
        return True
    ctx = (s.get("Context") or "").strip()
    dec = s.get("Decisions") or []
    act = s.get("Actions") or []
    ques = s.get("Questions") or []
    return (ctx == "") and (len(dec) == 0) and (len(act) == 0) and (len(ques) == 0)

def extract_summary_and_error(obj: Dict[str, Any]) -> Tuple[Dict[str, Any], Optional[str]]:
    """
    Rule:
    - If summary is a NON-empty dict -> normalize and accept (error=None unless normalization fails).
    - If summary is a dict but EMPTY -> try to recover from raw_summary:
        - if raw_summary parses -> use it (error=None)
        - else -> keep empty (error=real parsing error)
    - If summary missing/not dict -> try raw_summary:
        - if parses -> use it
        - else -> empty + error
    """
    s = obj.get("summary")

    if isinstance(s, dict) and not _is_effectively_empty_summary(s):
        normalized, err = parse_structured_summary(json.dumps(s, ensure_ascii=False))
        if err:
            return EMPTY_SUMMARY, err
        return normalized, None

    raw = obj.get("raw_summary")
    if isinstance(raw, str) and raw.strip():
        normalized, err = parse_structured_summary(raw)
        if err:
            return EMPTY_SUMMARY, err
        return normalized, None

    return EMPTY_SUMMARY, "no usable summary/raw_summary fields"

In [17]:
gold_items = json.loads(GOLD_PATH.read_text(encoding="utf-8"))
if not isinstance(gold_items, list):
    raise ValueError("gold_summaries_final.json must be a list of objects")

gold_by_id = {}
dialogue_by_id = {}
gold_parse_errors = []

for obj in gold_items:
    did = extract_dialogue_id(obj)
    dialogue_by_id[did] = extract_dialogue_text(obj)

    summ, err = extract_summary_and_error(obj)
    gold_by_id[did] = summ
    if err:
        gold_parse_errors.append({"dialogue_id": did, "error": err})

gold_ids = sorted(gold_by_id.keys())

print("gold:", len(gold_ids))
print("dialogues:", len(dialogue_by_id))
print("gold parse errors:", len(gold_parse_errors))

gold: 30
dialogues: 30
gold parse errors: 0


In [15]:
preds_by_model = {}
pred_parse_errors = []

for model_name, path in PRED_PATHS.items():
    items = load_jsonl(path)

    m = {}
    for obj in items:
        did = extract_dialogue_id(obj)

        if did not in dialogue_by_id:
            dtext = extract_dialogue_text(obj)
            if dtext:
                dialogue_by_id[did] = dtext

        summ, err = extract_summary_and_error(obj)
        m[did] = summ
        if err:
            pred_parse_errors.append({"model": model_name, "dialogue_id": did, "error": err})

    preds_by_model[model_name] = m

print({k: len(v) for k, v in preds_by_model.items()})
print("dialogues available:", len(dialogue_by_id))
print("pred parse errors:", len(pred_parse_errors))

pd.DataFrame(pred_parse_errors).head(10)

{'mistral_large': 30, 'ollama_llama3.1_8b': 30, 'ollama_mistral_7b': 30, 'ollama_qwen2.5_3b': 30, 'openai_gpt4o': 30}
dialogues available: 30
pred parse errors: 0


""


In [18]:
rows = []

for model_name, pred_map in preds_by_model.items():
    for did in gold_ids:
        ref = gold_by_id[did]
        gen = pred_map.get(did, {"Context": "", "Decisions": [], "Actions": [], "Questions": []})

        ctx_sim = context_similarity(EMBEDDER, ref["Context"], gen["Context"])

        dec = prf1_set(ref["Decisions"], gen["Decisions"])
        ques = prf1_set(ref["Questions"], gen["Questions"])

        for thr in ACTION_THRESHOLDS:
            act = match_actions_by_description(EMBEDDER, ref["Actions"], gen["Actions"], threshold=thr)

            rows.append({
                "model": model_name,
                "dialogue_id": did,
                "action_thr": thr,

                "context_sim": ctx_sim,

                "dec_p": dec["precision"], "dec_r": dec["recall"], "dec_f1": dec["f1"],
                "q_p": ques["precision"], "q_r": ques["recall"], "q_f1": ques["f1"],

                "act_p": act.precision, "act_r": act.recall, "act_f1": act.f1,
                "act_assignee_acc": act.assignee_acc,
                "act_deadline_acc": act.deadline_acc,

                "dec_tp": dec["tp"], "dec_fp": dec["fp"], "dec_fn": dec["fn"],
                "q_tp": ques["tp"], "q_fp": ques["fp"], "q_fn": ques["fn"],
                "act_tp": act.tp, "act_fp": act.fp, "act_fn": act.fn,

                "overall_auto": overall_auto_score(ctx_sim, act.f1, dec["f1"], ques["f1"]),
            })

df_auto = pd.DataFrame(rows)
df_auto.head()

,model,dialogue_id,action_thr,context_sim,dec_p,dec_r,dec_f1,q_p,q_r,q_f1,...,dec_tp,dec_fp,dec_fn,q_tp,q_fp,q_fn,act_tp,act_fp,act_fn,overall_auto
0,mistral_large,Chat_001__batch_00001,0.75,0.952481,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2,3,0,1,0,2,1,0,0.613368
1,mistral_large,Chat_001__batch_00001,0.80,0.952481,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2,3,0,1,0,2,1,0,0.613368
2,mistral_large,Chat_001__batch_00002,0.75,0.919706,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2,3,0,2,0,0,2,0,0.321897
3,mistral_large,Chat_001__batch_00002,0.80,0.919706,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2,3,0,2,0,0,2,0,0.321897
4,mistral_large,Chat_001__batch_00004,0.75,0.927471,0.0,0.0,0.0,0.0,0.0,0.0,...,0,4,0,0,2,0,7,0,4,0.596837


In [19]:
summary_rows = []

for (model, thr), g in df_auto.groupby(["model", "action_thr"]):
    # Context: macro mean
    ctx_mean = g["context_sim"].mean()

    # Macro F1 per dialogue
    dec_f1_macro = g["dec_f1"].mean()
    q_f1_macro = g["q_f1"].mean()
    act_f1_macro = g["act_f1"].mean()

    # Micro: sum TP/FP/FN
    dec_tp, dec_fp, dec_fn = int(g["dec_tp"].sum()), int(g["dec_fp"].sum()), int(g["dec_fn"].sum())
    q_tp, q_fp, q_fn = int(g["q_tp"].sum()), int(g["q_fp"].sum()), int(g["q_fn"].sum())
    act_tp, act_fp, act_fn = int(g["act_tp"].sum()), int(g["act_fp"].sum()), int(g["act_fn"].sum())

    _, _, dec_f1_micro = micro_prf(dec_tp, dec_fp, dec_fn)
    _, _, q_f1_micro = micro_prf(q_tp, q_fp, q_fn)
    _, _, act_f1_micro = micro_prf(act_tp, act_fp, act_fn)

    # Slots: macro mean over dialogues (on matched pairs)
    ass_mean = g["act_assignee_acc"].mean()
    ddl_mean = g["act_deadline_acc"].mean()

    overall_auto_mean = g["overall_auto"].mean()

    summary_rows.append({
        "model": model,
        "action_thr": thr,
        "context_sim_mean": ctx_mean,

        "actions_f1_micro": act_f1_micro,
        "actions_f1_macro": act_f1_macro,
        "actions_assignee_acc_mean": ass_mean,
        "actions_deadline_acc_mean": ddl_mean,

        "decisions_f1_micro": dec_f1_micro,
        "decisions_f1_macro": dec_f1_macro,

        "questions_f1_micro": q_f1_micro,
        "questions_f1_macro": q_f1_macro,

        "overall_auto_mean": overall_auto_mean,
    })

df_auto_summary = pd.DataFrame(summary_rows).sort_values(
    by=["action_thr","overall_auto_mean","actions_f1_micro","context_sim_mean"],
    ascending=[True, False, False, False]
)

df_auto_summary

,model,action_thr,context_sim_mean,actions_f1_micro,actions_f1_macro,actions_assignee_acc_mean,actions_deadline_acc_mean,decisions_f1_micro,decisions_f1_macro,questions_f1_micro,questions_f1_macro,overall_auto_mean
8,openai_gpt4o,0.75,0.933688,0.752941,0.683608,0.480833,0.616667,0.036364,0.400000,0.250000,0.493651,0.700101
2,ollama_llama3.1_8b,0.75,0.876466,0.446154,0.453519,0.233333,0.433333,0.000000,0.200000,0.000000,0.433333,0.560495
0,mistral_large,0.75,0.934993,0.692737,0.554540,0.326111,0.530556,0.018018,0.100000,0.055556,0.119048,0.554194
4,ollama_mistral_7b,0.75,0.875790,0.518519,0.454628,0.211111,0.394444,0.000000,0.266667,0.023256,0.077778,0.517313
6,ollama_qwen2.5_3b,0.75,0.837478,0.415385,0.409471,0.250000,0.338889,0.000000,0.366667,0.000000,0.166667,0.516432
9,openai_gpt4o,0.80,0.933688,0.752941,0.683608,0.480833,0.616667,0.036364,0.400000,0.250000,0.493651,0.700101
3,ollama_llama3.1_8b,0.80,0.876466,0.446154,0.453519,0.233333,0.433333,0.000000,0.200000,0.000000,0.433333,0.560495
1,mistral_large,0.80,0.934993,0.692737,0.554540,0.326111,0.530556,0.018018,0.100000,0.055556,0.119048,0.554194
5,ollama_mistral_7b,0.80,0.875790,0.518519,0.454628,0.211111,0.394444,0.000000,0.266667,0.023256,0.077778,0.517313
7,ollama_qwen2.5_3b,0.80,0.837478,0.415385,0.409471,0.250000,0.338889,0.000000,0.366667,0.000000,0.166667,0.516432


In [20]:
from openai import OpenAI
import time


client = OpenAI()

JUDGE_MODEL = "gpt-4.1-mini"  # or "gpt-4.1"

JUDGE_SYSTEM = """Ты строгий независимый эксперт по оценке саммари рабочих чатов.
Тебе дан диалог и сгенерированное саммари в JSON (Context/Decisions/Actions/Questions).

Оценивай по шкалам 0..10 (целые):
- usefulness: насколько это полезно для команды (что произошло, что решили, что делать дальше)
- coherence: связность и структурность, отсутствие мусора
- faithfulness: фактическая верность диалогу (нет выдумок, нет подмен, нет "додумываний")

Правила:
- НЕЛЬЗЯ повышать usefulness ценой выдумок. Любая галлюцинация снижает faithfulness.
- Если в саммари есть конкретные факты, которых в диалоге нет — это ошибка.
- Если ключевые решения/действия из диалога пропущены — снижай usefulness.

Верни ТОЛЬКО валидный JSON строго такого формата:
{
  "usefulness": 0,
  "coherence": 0,
  "faithfulness": 0,
  "overall": 0,
  "errors": ["..."]
}
"""

JUDGE_USER_TEMPLATE = """ДИАЛОГ:
{dialogue}

CANDIDATE SUMMARY (JSON):
{candidate_json}

Оцени candidate по правилам из system.
"""

EMPTY_SUMMARY = {"Context": "", "Decisions": [], "Actions": [], "Questions": []}

In [21]:
def judge_one(dialogue_text: str, candidate_summary: dict) -> dict:
    user = JUDGE_USER_TEMPLATE.format(
        dialogue=dialogue_text,
        candidate_json=json.dumps(candidate_summary, ensure_ascii=False),
    )
    resp = client.responses.create(
        model=JUDGE_MODEL,
        input=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user},
        ],
        temperature=0,
    )
    return json.loads(resp.output_text.strip())


JUDGE_CACHE_PATH = Path("../data/processed/judge_scores_protocolA.jsonl")
JUDGE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

In [22]:
def load_judge_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    cache = {}
    for obj in load_jsonl(path):
        # cache key includes judge_model (in case you change it later)
        key = (obj.get("judge_model"), obj.get("model"), obj.get("dialogue_id"))
        cache[key] = obj
    return cache


def append_judge_cache(path: Path, record: dict) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [23]:
judge_cache = load_judge_cache(JUDGE_CACHE_PATH)

judge_rows = []
judge_missing_candidates = []
judge_missing_dialogues = []

for model_name, pred_map in preds_by_model.items():
    for did in gold_ids:
        cache_key = (JUDGE_MODEL, model_name, did)
        if cache_key in judge_cache:
            judge_rows.append(judge_cache[cache_key])
            continue

        dialogue = dialogue_by_id.get(did, "")
        if not dialogue.strip():
            judge_missing_dialogues.append({"model": model_name, "dialogue_id": did})
            continue

        cand = pred_map.get(did)
        if cand is None:
            judge_missing_candidates.append({"model": model_name, "dialogue_id": did})
            cand = EMPTY_SUMMARY

        try:
            j = judge_one(dialogue, cand)
            rec = {
                "judge_model": JUDGE_MODEL,
                "model": model_name,
                "dialogue_id": did,
                **j,
            }
            append_judge_cache(JUDGE_CACHE_PATH, rec)
            judge_rows.append(rec)
            time.sleep(0.2)

        except Exception as e:
            rec = {
                "judge_model": JUDGE_MODEL,
                "model": model_name,
                "dialogue_id": did,
                "error": str(e),
            }
            append_judge_cache(JUDGE_CACHE_PATH, rec)
            judge_rows.append(rec)

print("judge rows:", len(judge_rows))
print("judge missing dialogues:", len(judge_missing_dialogues))
print("judge missing candidates:", len(judge_missing_candidates))

pd.DataFrame(judge_missing_candidates).head(10), pd.DataFrame(judge_missing_dialogues).head(10)

judge rows: 150
judge missing dialogues: 0
judge missing candidates: 0


(Empty DataFrame
 Columns: []
 Index: [],
 Empty DataFrame
 Columns: []
 Index: [])

In [30]:
df_judge = pd.DataFrame(judge_rows)
if "error" in df_judge.columns:
    df_judge_ok = df_judge[df_judge["error"].isna()].copy()
else:
    df_judge_ok = df_judge.copy()

df_judge_summary = df_judge_ok.groupby("model").agg(
    judge_usefulness_mean=("usefulness", "mean"),
    judge_coherence_mean=("coherence", "mean"),
    judge_faithfulness_mean=("faithfulness", "mean"),
    judge_overall_mean=("overall", "mean"),
    n=("overall", "count"),
).reset_index().sort_values("judge_overall_mean", ascending=False)

df_judge_summary

,model,judge_usefulness_mean,judge_coherence_mean,judge_faithfulness_mean,judge_overall_mean,n
0,mistral_large,8.600000,8.966667,9.533333,8.833333,30
4,openai_gpt4o,7.466667,8.366667,9.533333,8.266667,30
2,ollama_mistral_7b,7.066667,7.933333,9.033333,7.800000,30
1,ollama_llama3.1_8b,6.833333,7.833333,9.033333,7.733333,30
3,ollama_qwen2.5_3b,6.266667,7.300000,8.533333,7.133333,30


In [31]:
final = df_auto_summary.merge(df_judge_summary, on="model", how="left")

final_sorted = final.sort_values(
    by=["action_thr", "judge_faithfulness_mean", "judge_usefulness_mean", "overall_auto_mean"],
    ascending=[True, False, False, False]
)

final_sorted

,model,action_thr,context_sim_mean,actions_f1_micro,actions_f1_macro,actions_assignee_acc_mean,actions_deadline_acc_mean,decisions_f1_micro,decisions_f1_macro,questions_f1_micro,questions_f1_macro,overall_auto_mean,judge_usefulness_mean,judge_coherence_mean,judge_faithfulness_mean,judge_overall_mean,n
2,mistral_large,0.75,0.934993,0.692737,0.554540,0.326111,0.530556,0.018018,0.100000,0.055556,0.119048,0.554194,8.600000,8.966667,9.533333,8.833333,30
0,openai_gpt4o,0.75,0.933688,0.752941,0.683608,0.480833,0.616667,0.036364,0.400000,0.250000,0.493651,0.700101,7.466667,8.366667,9.533333,8.266667,30
3,ollama_mistral_7b,0.75,0.875790,0.518519,0.454628,0.211111,0.394444,0.000000,0.266667,0.023256,0.077778,0.517313,7.066667,7.933333,9.033333,7.800000,30
1,ollama_llama3.1_8b,0.75,0.876466,0.446154,0.453519,0.233333,0.433333,0.000000,0.200000,0.000000,0.433333,0.560495,6.833333,7.833333,9.033333,7.733333,30
4,ollama_qwen2.5_3b,0.75,0.837478,0.415385,0.409471,0.250000,0.338889,0.000000,0.366667,0.000000,0.166667,0.516432,6.266667,7.300000,8.533333,7.133333,30
7,mistral_large,0.80,0.934993,0.692737,0.554540,0.326111,0.530556,0.018018,0.100000,0.055556,0.119048,0.554194,8.600000,8.966667,9.533333,8.833333,30
5,openai_gpt4o,0.80,0.933688,0.752941,0.683608,0.480833,0.616667,0.036364,0.400000,0.250000,0.493651,0.700101,7.466667,8.366667,9.533333,8.266667,30
8,ollama_mistral_7b,0.80,0.875790,0.518519,0.454628,0.211111,0.394444,0.000000,0.266667,0.023256,0.077778,0.517313,7.066667,7.933333,9.033333,7.800000,30
6,ollama_llama3.1_8b,0.80,0.876466,0.446154,0.453519,0.233333,0.433333,0.000000,0.200000,0.000000,0.433333,0.560495,6.833333,7.833333,9.033333,7.733333,30
9,ollama_qwen2.5_3b,0.80,0.837478,0.415385,0.409471,0.250000,0.338889,0.000000,0.366667,0.000000,0.166667,0.516432,6.266667,7.300000,8.533333,7.133333,30


In [ ]:
MODEL = final_sorted["model"].iloc[0]

bad = df_judge_ok[df_judge_ok["model"] == MODEL].sort_values("faithfulness").head(5)
bad

,judge_model,model,dialogue_id,usefulness,coherence,faithfulness,overall,errors
6,gpt-4.1-mini,mistral_large,Chat_001__batch_00014,8,9,7,8,[В решениях указано использование API Google G...
21,gpt-4.1-mini,mistral_large,Chat_002__batch_00010,8,9,9,8,[]
15,gpt-4.1-mini,mistral_large,Chat_001__batch_00066,9,9,9,9,[]
28,gpt-4.1-mini,mistral_large,Chat_003__batch_00000,9,9,9,9,[]
19,gpt-4.1-mini,mistral_large,Chat_002__batch_00005,8,9,9,8,[]


In [27]:
def _safe_len(x):
    return int(len(x)) if isinstance(x, list) else 0

def _count_actions(actions):
    return int(sum(1 for a in (actions or []) if isinstance(a, dict) and (a.get("description") or "").strip()))

# 1) Считаем "verbosity" по кандидатам и "coverage baseline" по gold (для контекста)
cov_rows = []
for model_name, pred_map in preds_by_model.items():
    for did in gold_ids:
        ref = gold_by_id[did]
        gen = pred_map.get(did, {"Context": "", "Decisions": [], "Actions": [], "Questions": []})

        cov_rows.append({
            "model": model_name,
            "dialogue_id": did,

            # Gold counts (baseline: сколько "должно" быть)
            "gold_dec_n": _safe_len(ref.get("Decisions")),
            "gold_act_n": _count_actions(ref.get("Actions")),
            "gold_q_n": _safe_len(ref.get("Questions")),

            # Candidate counts (verbosity)
            "gen_dec_n": _safe_len(gen.get("Decisions")),
            "gen_act_n": _count_actions(gen.get("Actions")),
            "gen_q_n": _safe_len(gen.get("Questions")),
        })

df_cov = pd.DataFrame(cov_rows)

In [28]:
# 2) Берём micro precision/recall из df_auto (по Decisions/Questions) + Actions из выбранного порога
# (Чтобы соответствовало твоей итоговой таблице, выбери основной порог)
MAIN_ACTION_THR = 0.80
df_main = df_auto[df_auto["action_thr"] == MAIN_ACTION_THR].copy()

# micro P/R для set-метрик считаем из tp/fp/fn
def micro_pr(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    return p, r

rows = []
for model, g in df_main.groupby("model"):
    # micro PR: Decisions
    dec_tp, dec_fp, dec_fn = int(g["dec_tp"].sum()), int(g["dec_fp"].sum()), int(g["dec_fn"].sum())
    dec_p, dec_r = micro_pr(dec_tp, dec_fp, dec_fn)

    # micro PR: Questions
    q_tp, q_fp, q_fn = int(g["q_tp"].sum()), int(g["q_fp"].sum()), int(g["q_fn"].sum())
    q_p, q_r = micro_pr(q_tp, q_fp, q_fn)

    # micro PR: Actions (по matched actions)
    act_tp, act_fp, act_fn = int(g["act_tp"].sum()), int(g["act_fp"].sum()), int(g["act_fn"].sum())
    act_p, act_r = micro_pr(act_tp, act_fp, act_fn)

    # verbosity: средние количества items
    gg = df_cov[df_cov["model"] == model]
    rows.append({
        "model": model,
        "action_thr": MAIN_ACTION_THR,

        # baseline (gold)
        "gold_dec_avg": gg["gold_dec_n"].mean(),
        "gold_act_avg": gg["gold_act_n"].mean(),
        "gold_q_avg": gg["gold_q_n"].mean(),

        # verbosity (gen)
        "gen_dec_avg": gg["gen_dec_n"].mean(),
        "gen_act_avg": gg["gen_act_n"].mean(),
        "gen_q_avg": gg["gen_q_n"].mean(),

        # micro precision/recall
        "dec_p_micro": dec_p,
        "dec_r_micro": dec_r,
        "act_p_micro": act_p,
        "act_r_micro": act_r,
        "q_p_micro": q_p,
        "q_r_micro": q_r,

        # helpful ratios (who is "silent" vs "verbose")
        "dec_gen_to_gold_ratio": (gg["gen_dec_n"].sum() / max(1, gg["gold_dec_n"].sum())),
        "act_gen_to_gold_ratio": (gg["gen_act_n"].sum() / max(1, gg["gold_act_n"].sum())),
        "q_gen_to_gold_ratio": (gg["gen_q_n"].sum() / max(1, gg["gold_q_n"].sum())),
    })

df_cov_report = pd.DataFrame(rows).sort_values(
    by=["act_r_micro", "act_p_micro", "gen_act_avg"],
    ascending=[False, False, True]
)

df_cov_report

,model,action_thr,gold_dec_avg,gold_act_avg,gold_q_avg,gen_dec_avg,gen_act_avg,gen_q_avg,dec_p_micro,dec_r_micro,act_p_micro,act_r_micro,q_p_micro,q_r_micro,dec_gen_to_gold_ratio,act_gen_to_gold_ratio,q_gen_to_gold_ratio
4,openai_gpt4o,0.8,1.3,2.966667,0.866667,0.533333,2.700000,1.000000,0.062500,0.025641,0.790123,0.719101,0.233333,0.269231,0.410256,0.910112,1.153846
0,mistral_large,0.8,1.3,2.966667,0.866667,2.400000,3.000000,1.533333,0.013889,0.025641,0.688889,0.696629,0.043478,0.076923,1.846154,1.011236,1.769231
2,ollama_mistral_7b,0.8,1.3,2.966667,0.866667,1.400000,2.433333,2.000000,0.000000,0.000000,0.575342,0.471910,0.016667,0.038462,1.076923,0.820225,2.307692
1,ollama_llama3.1_8b,0.8,1.3,2.966667,0.866667,1.000000,1.366667,0.466667,0.000000,0.000000,0.707317,0.325843,0.000000,0.000000,0.769231,0.460674,0.538462
3,ollama_qwen2.5_3b,0.8,1.3,2.966667,0.866667,0.900000,1.366667,1.233333,0.000000,0.000000,0.658537,0.303371,0.000000,0.000000,0.692308,0.460674,1.423077


In [29]:
def flag_silent_verbose(row, ratio_low=0.7, ratio_high=1.3):
    flags = []
    if row["dec_gen_to_gold_ratio"] < ratio_low and row["dec_r_micro"] < 0.6:
        flags.append("decisions_silent")
    if row["act_gen_to_gold_ratio"] < ratio_low and row["act_r_micro"] < 0.6:
        flags.append("actions_silent")
    if row["q_gen_to_gold_ratio"] < ratio_low and row["q_r_micro"] < 0.6:
        flags.append("questions_silent")

    if row["dec_gen_to_gold_ratio"] > ratio_high and row["dec_p_micro"] < 0.6:
        flags.append("decisions_verbose")
    if row["act_gen_to_gold_ratio"] > ratio_high and row["act_p_micro"] < 0.6:
        flags.append("actions_verbose")
    if row["q_gen_to_gold_ratio"] > ratio_high and row["q_p_micro"] < 0.6:
        flags.append("questions_verbose")

    return ", ".join(flags)

df_cov_report["flags"] = df_cov_report.apply(flag_silent_verbose, axis=1)
df_cov_report[["model","gold_act_avg","gen_act_avg","act_p_micro","act_r_micro","act_gen_to_gold_ratio","flags"]]

,model,gold_act_avg,gen_act_avg,act_p_micro,act_r_micro,act_gen_to_gold_ratio,flags
4,openai_gpt4o,2.966667,2.700000,0.790123,0.719101,0.910112,decisions_silent
0,mistral_large,2.966667,3.000000,0.688889,0.696629,1.011236,"decisions_verbose, questions_verbose"
2,ollama_mistral_7b,2.966667,2.433333,0.575342,0.471910,0.820225,questions_verbose
1,ollama_llama3.1_8b,2.966667,1.366667,0.707317,0.325843,0.460674,"actions_silent, questions_silent"
3,ollama_qwen2.5_3b,2.966667,1.366667,0.658537,0.303371,0.460674,"decisions_silent, actions_silent, questions_ve..."
